# Axis III, RQ1 -- 50 subjects (TotalSegmentatorV2)

Implements **only RQ1** on the TotalSegmentatorV2 50-subject dataset. This notebook runs
**the first 50 loaded subjects** in one pass -- no batching.

**RQ1 -- Do the 32 text-image fusion channels distinguish left and right anatomical structures?**

Uses the **TotalSegmentatorV2 50-subject dataset** (`Fatema142/totalsegmentator-50-rq-analysis`
on Hugging Face).

The VoxTell setup (environment install, model loading, Qwen3 text-encoder fix) pins whatever
numpy/scipy/torch/transformers/etc. Kaggle's kernel already has loaded via a pip constraints
file, installs voxtell with `--no-deps`, and only adds the runtime deps actually missing
(nnunetv2, acvl-utils, etc.). Nothing already-loaded-in-memory gets replaced on disk, so no
kernel restart is needed and "Save & Run All" works end to end.

| RQ | Question (short) | Folder | GPU cost |
|---|---|---|---|
| RQ1 | Do the 32 fusion channels distinguish left/right? | `results/rq1/` | 1 forward pass/side/organ x 4 stages |


## 1. Check GPU

In [ ]:
!nvidia-smi


## 2. Setup working directories (Kaggle environment)

In [ ]:
import os
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
RESULTS_ROOT = WORKING_DIR / 'results'
for _rq in ('rq1', 'rq2', 'rq3', 'rq4', 'rq5', 'rq6', 'rq7', 'rq8', 'rq9', 'rq10', 'rq11'):
    (RESULTS_ROOT / _rq).mkdir(parents=True, exist_ok=True)
print('Results will be saved under:', RESULTS_ROOT, 'in subfolders rq1/ ... rq11/')


## 3. Install VoxTell

In [ ]:
import subprocess, sys, importlib
from importlib import metadata
from pathlib import Path
import os

# Reduce CUDA memory fragmentation (must be set before the first CUDA allocation)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Installs VoxTell + its missing dependencies WITHOUT touching any compiled package that
# Kaggle's kernel already has loaded in memory (numpy, scipy, torch, ...). Replacing those
# on disk mid-session is what caused errors like:
#   ImportError: cannot import name '_slice' from 'numpy._core.umath'
# because a running process cannot swap a loaded C-extension. By pinning them to the
# versions already installed, no kernel restart is needed and "Save Version" works.
#
# voxtell>=0.1.2 accepts transformers<6, so Kaggle's preinstalled transformers is kept too.

_PIN_IF_PRESENT = [
    'numpy', 'scipy', 'torch', 'torchvision', 'torchaudio', 'pandas', 'scikit-learn',
    'scikit-image', 'pillow', 'matplotlib', 'transformers', 'tokenizers',
    'huggingface-hub', 'safetensors', 'nibabel', 'SimpleITK', 'tqdm', 'requests',
]

def _installed(pkg):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None

try:
    _vt_version = metadata.version('voxtell')
except metadata.PackageNotFoundError:
    _vt_version = None

if _vt_version:
    print(f'voxtell {_vt_version} already installed -- skipping pip setup.')
else:
    _constraints = Path('/kaggle/working/.pip_constraints.txt')
    _pins = [f'{p}=={_installed(p)}' for p in _PIN_IF_PRESENT if _installed(p)]
    _constraints.write_text('\n'.join(_pins) + '\n')
    print('Pinned (kept as-is):', ', '.join(_pins))

    def _pip(*args):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                               '-c', str(_constraints), *args])

    # 1) voxtell itself without deps (its torch!=2.9 metadata must not trigger a torch swap)
    _pip('--no-deps', 'voxtell>=0.1.2')
    # 2) only the runtime deps that are actually missing; pinned packages stay untouched
    _pip('nnunetv2>=2.7', 'acvl-utils>=0.2.3,<0.3', 'dynamic-network-architectures>=0.4.1,<0.5',
         'positional-encodings', 'einops', 'SimpleITK', 'nibabel', 'tqdm', 'huggingface_hub')

    importlib.invalidate_caches()
    print('voxtell installed:', metadata.version('voxtell'), '-- no restart needed.')


## 4. Clone the research repo

Uses a Kaggle Secret named `GITHUB_PAT` (Add-ons -> Secrets) -- never hardcode a token here
(an earlier sibling notebook did, and it had to be flagged and rotated).

In [ ]:
import os
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

_gh_token = UserSecretsClient().get_secret("GITHUB_PAT")
assert _gh_token, "Add a Kaggle Secret named GITHUB_PAT before running this cell."

_clone_url = f'https://{_gh_token}@github.com/fatemafaria142/positional-bias-medical-vlms.git'
_dest = '/kaggle/working/research_repo'
shutil.rmtree(_dest, ignore_errors=True)

_result = subprocess.run(
    ['git', 'clone', '--branch', 'teamwork/totalsegmentatorV2', '--depth', '1', _clone_url, _dest],
    capture_output=True, text=True,
)
_stderr_scrubbed = _result.stderr.replace(_gh_token, '***')
del _gh_token, _clone_url

if _result.returncode != 0:
    raise RuntimeError(f'git clone failed (exit {_result.returncode}):\n{_stderr_scrubbed}')

RESEARCH_REPO = Path(_dest)
print('cloned into', RESEARCH_REPO)


## 4b. Download the 50-subject TotalSegmentatorV2 dataset from Hugging Face

Public dataset repo (`Fatema142/totalsegmentator-50-rq-analysis`) -- no Kaggle Secret / token
needed to download it.


In [ ]:
from huggingface_hub import snapshot_download

TOTALSEG_DIR = Path(snapshot_download(
    repo_id='Fatema142/totalsegmentator-50-rq-analysis',
    repo_type='dataset',
    local_dir=str(WORKING_DIR / 'totalsegmentator_50'),
))
print('downloaded to', TOTALSEG_DIR)


## 5. Download and load the VoxTell model

In [ ]:
import torch
from huggingface_hub import snapshot_download
from voxtell.inference.predictor import VoxTellPredictor

VOXTELL_MODEL_VERSION = 'voxtell_v1.1'
CKPT_CACHE_DIR = WORKING_DIR / 'checkpoints'
CKPT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_local_repo_dir = snapshot_download(
    repo_id='mrokuss/VoxTell',
    allow_patterns=[f'{VOXTELL_MODEL_VERSION}/*'],
    local_dir=str(CKPT_CACHE_DIR),
)
VOXTELL_MODEL_DIR = str(Path(_local_repo_dir) / VOXTELL_MODEL_VERSION)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
import voxtell.inference.predictor as _vp

class _NoQwen:  # VoxTellPredictor.__init__ would load a full fp32 Qwen3-4B (~16 GB) into host RAM; cell 14
    @staticmethod  # then loads the fp16 one we actually use. Holding both at once can kill the Kaggle kernel.
    def from_pretrained(*a, **k):
        return torch.nn.Identity()

_orig_automodel = _vp.AutoModel
_vp.AutoModel = _NoQwen
try:
    predictor = VoxTellPredictor(model_dir=VOXTELL_MODEL_DIR, device=device)
finally:
    _vp.AutoModel = _orig_automodel
# VoxTellPredictor.__init__ loads the checkpoint on CPU and only moves the network to
# `device` inside predict_sliding_window_return_logits() -- every mechanistic helper below
# calls model.encoder/model.decoder directly (bypassing that method), so the network must be
# moved here or every such call hits 'Input type ... and weight type ... should be the same'.
predictor.network = predictor.network.to(device)
model = predictor.network
print('VoxTell predictor ready on', device, '| num_heads (RQ1 fusion channels):', model.num_heads)
assert model.num_heads == 32, f'RQ1 assumes 32 fusion channels, checkpoint has num_heads={model.num_heads}'


## 5b. Fix: text encoder OOM on a single T4/P100 (encoder on CPU + embedding cache)

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

if not hasattr(predictor, '_orig_embed_text_prompts'):
    predictor._orig_embed_text_prompts = predictor.embed_text_prompts

# The 4B-param fp16 text encoder (~8 GB) is kept on the CPU, NOT on the GPU: on a 15 GB T4 it left too
# little room for the segmentation network (RQ3's run_interventions died with "out of GPU memory even
# with one output per pass"). Embeddings are cached per prompt string, so each distinct prompt is
# computed once (briefly moving the encoder to the GPU) and every later call is a dictionary lookup --
# no 8 GB CPU<->GPU transfer per call like the original VoxTellPredictor.embed_text_prompts.
predictor.text_backbone = AutoModel.from_pretrained(
    'Qwen/Qwen3-Embedding-4B', dtype=torch.float16  # VoxTell v0.1.0's default text encoder
).eval()
if getattr(predictor, 'tokenizer', None) is None:
    predictor.tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-4B')

import threading
from voxtell.utils.text_embedding import last_token_pool, wrap_with_instruction

_EMBED_CACHE = {}  # prompt string -> (D,) float32 CPU tensor
_embed_cache_lock = threading.Lock()


@torch.inference_mode()
def _embed_missing(prompts):
    """Embed `prompts` (list of str) with the encoder moved onto the GPU just for this call."""
    dev = predictor.device
    torch.cuda.empty_cache()
    predictor.text_backbone.to(dev)
    try:
        wrapped = wrap_with_instruction(list(prompts))
        tok = predictor.tokenizer(wrapped, padding=True, truncation=True,
                                  max_length=predictor.max_text_length, return_tensors='pt')
        tok = {k: v.to(dev) for k, v in tok.items()}
        out = predictor.text_backbone(**tok)
        emb = last_token_pool(out.last_hidden_state, tok['attention_mask']).float().cpu()
    finally:
        predictor.text_backbone.to('cpu')
        torch.cuda.empty_cache()
    return emb


def _embed_text_prompts_fp16_safe(text_prompts):
    """Same output as VoxTellPredictor.embed_text_prompts: (1, n_prompts, D) float32 on predictor.device."""
    if isinstance(text_prompts, str):
        text_prompts = [text_prompts]
    text_prompts = list(text_prompts)
    with _embed_cache_lock:
        missing = list(dict.fromkeys(t for t in text_prompts if t not in _EMBED_CACHE))
        if missing:
            for t, e in zip(missing, _embed_missing(missing)):
                _EMBED_CACHE[t] = e
        emb = torch.stack([_EMBED_CACHE[t] for t in text_prompts])
    return emb.view(1, len(text_prompts), -1).to(predictor.device)


predictor.embed_text_prompts = _embed_text_prompts_fp16_safe
print('text encoder: fp16 on CPU + per-prompt embedding cache; tokenizer loaded:',
      predictor.tokenizer is not None)


## 6. Helpers (I/O, manifest, RAS axis bookkeeping -- same logic as `scripts/common3d.py`)

In [ ]:
import csv
from pathlib import Path

import numpy as np
from nnunetv2.imageio.nibabel_reader_writer import NibabelIOWithReorient

RESEARCH_REPO = Path(RESEARCH_REPO)
reader = NibabelIOWithReorient()

def load_volume_ras(path):
    """Load a NIfTI image via VoxTell's own reader so predicted masks and ground truth end up
    in the exact same RAS-reoriented array space. Returns (1, X, Y, Z)."""
    data, props = reader.read_images([str(path)])
    return np.asarray(data), props

def spatial_view(arr_4d):
    a = np.asarray(arr_4d)
    while a.ndim > 3 and a.shape[0] == 1:
        a = a[0]
    return a

def load_seg_ras(path):
    data, props = reader.read_seg(str(path))
    return spatial_view(data).astype(np.int32), props

# NibabelIOWithReorient returns axes (Superior-Inferior, Anterior-Posterior, Right-Left) --
# verified empirically against real predictions (left/right kidney centroids differ ~85 voxels
# on this axis, ~3 on the others, for every subject).
AXCODES = ("S", "A", "R")
RL_AXIS = AXCODES.index("R")

def dice_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2.0 * inter / denom)

def iou_score(pred, gt):
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return 1.0 if union == 0 else float(inter / union)

print('helpers ready')


## 7. Load the experiment code

Imports `scripts/voxtell_mechanistic.py` (exports `encode_mask_rle`/`decode_mask_rle`/
`encode_array_b64`/`decode_array_b64`, the shared CSV-persistence helpers used by every RQ
below) and `scripts/voxtell_experiments.py` (`run_interventions(..., return_masks=True)`,
reused directly by RQ3/RQ4) from the cloned repo, via `sys.path`.

In [ ]:
import sys

sys.path.insert(0, str(RESEARCH_REPO / 'scripts'))
import voxtell_mechanistic as vm
import voxtell_experiments as ve

print('loaded voxtell_mechanistic from', vm.__file__)
print('loaded voxtell_experiments from', ve.__file__)
assert hasattr(vm, 'encode_mask_rle'), 'need the CSV-persistence helpers in voxtell_mechanistic.py'
assert 'return_masks' in ve.run_interventions.__code__.co_varnames, 'need run_interventions(return_masks=True)'


## 8. Subjects and organs -- ALL 50 staged subjects x every bilateral organ pair

`SUBJECTS` is discovered from the manifest itself (every distinct `subject_id` staged by
`scripts/stage_totalsegmentator_50.py` into `Fatema142/totalsegmentator-50-rq-analysis` -- all 50
subjects, not a hand-picked subset), rather than hardcoded, so this always matches whatever is
actually in the downloaded dataset. `ORGAN_CORES = None` keeps every bilateral organ pair
present in the manifest -- the 19-pair TotalSegmentatorV2 list in
`scripts/stage_totalsegmentator_50.py::BILATERAL_ORGAN_STEMS` (adrenal gland, autochthon,
clavicle, common carotid artery, brachiocephalic vein, femur, gluteus maximus/medius/minimus,
hip, humerus, iliac artery, iliac vein, iliopsoas, kidney, kidney cyst, lung lower/upper lobe,
scapula, subclavian artery). Unilateral structures (lung middle lobe, ribs, vertebrae, skull,
etc.) have no left/right counterpart under this project's scope and are dropped automatically.

**TotalSegmentatorV2 stores one separate binary mask file per organ/side**
(`segmentations/<organ>_<side>.nii.gz` in the source release) -- `load_subject` below assembles
those into one synthetic multi-label volume (a fresh integer id per organ/side, assigned from
the manifest) so every RQ function in `scripts/voxtell_experiments.py`/`voxtell_mechanistic.py`
can work directly on it (they expect a single multi-label `label_vol` + an `ids` dict).

**Running all RQs over all 50 subjects in one sitting is a lot of GPU time.** If a session times
out, lower `MAX_SUBJECTS` below to a smaller number first, confirm the full notebook runs end to
end, then remove the cap and let it run the full 50 (e.g. across several Kaggle sessions,
appending to the same CSVs since `append_rows` already skips subjects already on disk).


In [ ]:
DATASET_NAME = 'totalsegmentator_v2'
PERSON = 'faria'
ORGAN_CORES = None  # None = every bilateral organ pair present in the manifest
MAX_SUBJECTS = None  # None = all 50 staged subjects; set to e.g. 10 for a quick dry run


def load_manifest_totalsegmentator_50():
    """Loads the 50-subject TotalSegmentatorV2 manifest from the Hugging Face-downloaded
    dataset dir (TOTALSEG_DIR, set in cell 4b). Each row's `organ` column has NO side prefix
    ('adrenal gland', not 'left adrenal gland') -- side lives in its own `side` column -- and
    there is no shared `label_id`: every organ/side has its OWN mask file (`label_path`), so
    `load_subject` below builds one synthetic multi-label volume per subject."""
    path = TOTALSEG_DIR / 'manifest_totalsegmentator_50.csv'
    with open(path, newline='') as f:
        return list(csv.DictReader(f))


_all_rows = load_manifest_totalsegmentator_50()
for _r in _all_rows:
    _r['organ_full'] = f"{_r['side']} {_r['organ']}"  # 'left adrenal gland' -- the 'side organ' form ve.mirror() expects
_all_subject_ids = sorted({r['subject_id'] for r in _all_rows})
assert len(_all_subject_ids) == 50, f'expected 50 staged subjects, manifest has {len(_all_subject_ids)}'
SUBJECTS = [(PERSON, sid) for sid in (_all_subject_ids if MAX_SUBJECTS is None else _all_subject_ids[:MAX_SUBJECTS])]
print(f'using {len(SUBJECTS)} of {len(_all_subject_ids)} staged subjects')

subject_organs = {}
for person, sid in SUBJECTS:
    rows = [r for r in _all_rows if r['subject_id'] == sid]
    if ORGAN_CORES:
        rows = [r for r in rows if r['organ'] in ORGAN_CORES]
    have = {r['organ_full'] for r in rows}
    rows = [r for r in rows if ve.mirror(r['organ_full']) in have]  # keep complete left/right pairs only
    assert rows, f'no bilateral organs for {person}/{sid}'
    # TotalSegmentator has no shared label_id (one mask file per organ/side) -- assign one here,
    # deterministically (sorted by organ_full), so label_vol can be built in load_subject below
    for _i, r in enumerate(sorted(rows, key=lambda r: r['organ_full']), start=1):
        r['label_id'] = _i
    subject_organs[sid] = {'person': person, 'rows': rows}

n_pairs = {sid: len({r['organ'] for r in v['rows']}) for sid, v in subject_organs.items()}
print(len(subject_organs), 'subjects; bilateral organ pairs per subject (first 5 shown):',
      dict(list(n_pairs.items())[:5]))


def load_subject(sid):
    """Returns (person, img_arr, label_vol, organs, ids, prompts) -- the interface every RQ
    function in scripts/voxtell_experiments.py and voxtell_mechanistic.py expects. label_vol is
    assembled here from TotalSegmentator's separate per-organ/side binary mask files (one
    nib.load per organ/side, ~20-40 per subject) into one synthetic multi-label volume."""
    info = subject_organs[sid]
    rows = info['rows']
    base = TOTALSEG_DIR
    img_arr, _ = load_volume_ras(base / rows[0]['image_path'])
    label_vol = None
    for r in rows:
        seg, _ = load_seg_ras(base / r['label_path'])
        if label_vol is None:
            label_vol = np.zeros(seg.shape, dtype=np.int32)
        label_vol[seg > 0] = r['label_id']  # last-writer-wins on overlap (none expected between distinct organs)
    organs = [r['organ_full'] for r in rows]
    ids = {r['organ_full']: r['label_id'] for r in rows}
    prompts = {r['organ_full']: r['prompt_text'] for r in rows}
    return info['person'], img_arr, label_vol, organs, ids, prompts


## 8b. Resumable, crash-safe CSV writer + dual-GPU dispatch

Every RQ loop below writes each subject's rows to CSV **immediately** (not once at the end), and skips subjects already present in its output CSV on startup -- so a Kaggle session timeout/restart/crash loses at most the subject(s) in flight, and rerunning a cell picks up where it left off instead of starting over. If 2 GPUs are visible (Kaggle accelerator "GPU T4 x2"), subjects are split across both and processed concurrently via a second full predictor+model loaded onto the second device.

In [ ]:
import csv
import os
import queue
import threading
import time
import traceback
from pathlib import Path

# Use every visible GPU, not just predictor.device -- set the Kaggle accelerator to "GPU T4 x2"
# (Settings -> Accelerator) to actually get a second device here; with a single-GPU accelerator
# this safely falls back to running everything on predictor.device as before.
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0
DEVICES = [torch.device(f'cuda:{i}') for i in range(N_GPUS)] if N_GPUS else [predictor.device]
# Safe default: ONE GPU. Building a second VoxTell predictor on cuda:1 is the heaviest step of this cell
# (host RAM + GPU memory) and is what has been killing the Kaggle kernel. Set True only once everything
# runs cleanly on one GPU and you want the extra speed.
USE_SECOND_GPU = False
if not USE_SECOND_GPU:
    DEVICES = [predictor.device]
print('visible CUDA devices:', DEVICES)

PREDICTORS_BY_DEVICE = {predictor.device: predictor}
MODELS_BY_DEVICE = {predictor.device: model}


_embed_lock = threading.Lock()


def _build_predictor_on(device):
    """A second, fully independent VoxTellPredictor (segmentation network + its own
    Qwen3-Embedding-4B text backbone) on `device`, so the RQ loops below can process different
    subjects on different GPUs at the same time instead of everything funneling through the one
    predictor on cuda:0. Uses the same fp16 text backbone and on-device-resident embed_text_prompts
    as cell 14, just bound to this device instead."""
    from voxtell.inference.predictor import VoxTellPredictor
    import voxtell.inference.predictor as _vp

    class _NoQwen:  # the constructor would otherwise load a full fp32 Qwen3-4B (~16 GB) into host RAM,
        @staticmethod  # which can kill the Kaggle kernel; we never use that copy anyway (see below)
        def from_pretrained(*a, **k):
            return torch.nn.Identity()

    _orig_automodel = _vp.AutoModel
    _vp.AutoModel = _NoQwen
    try:
        p = VoxTellPredictor(model_dir=VOXTELL_MODEL_DIR, device=device)
    finally:
        _vp.AutoModel = _orig_automodel
    p.network = p.network.to(device)
    # NO second Qwen3 copy on this GPU: a 4B fp16 text encoder is ~8 GB, and on a 15 GB T4 that
    # left no room for the segmentation network's activations (CUDA OOM on every subject). The text
    # encoder is cheap, so all devices share the ONE on the main predictor (serialized by a lock)
    # and the resulting embedding is moved onto this device. The CPU copy VoxTellPredictor's
    # constructor loads is dropped too (frees ~16 GB of host RAM).
    p.text_backbone = None
    main = PREDICTORS_BY_DEVICE[predictor.device]

    def _embed(text_prompts):
        with _embed_lock:
            return main.embed_text_prompts(text_prompts).to(device)

    p.embed_text_prompts = _embed
    return p


import gc
gc.collect()  # drop tensors/graphs still referenced by earlier failed runs (e.g. via stored tracebacks)
for _device in DEVICES:
    if _device.type == 'cuda':
        with torch.cuda.device(_device):
            torch.cuda.empty_cache()
        _free, _total = torch.cuda.mem_get_info(_device)
        print(f'{_device}: {_free / 2**30:.1f} / {_total / 2**30:.1f} GiB free before loading')
        if _device not in PREDICTORS_BY_DEVICE and _free < 8 * 2**30:
            raise RuntimeError(f'{_device} only has {_free / 2**30:.1f} GiB free -- something from an earlier '
                               'run is still holding GPU memory. Restart the kernel (Run -> Restart) and re-run '
                               'the setup cells; re-running this cell cannot free it.')

for _device in DEVICES:
    if _device not in PREDICTORS_BY_DEVICE:
        print('loading a second VoxTell + Qwen3 text encoder on', _device, '...')
        PREDICTORS_BY_DEVICE[_device] = _build_predictor_on(_device)
    MODELS_BY_DEVICE[_device] = PREDICTORS_BY_DEVICE[_device].network

WORKER_DEVICES = list(PREDICTORS_BY_DEVICE.keys())
print(f'ready: {len(WORKER_DEVICES)} worker device(s) ->', WORKER_DEVICES)

_csv_lock = threading.Lock()


def completed_ids(csv_path, id_field='subject_id'):
    """Subject IDs already present in csv_path -- lets a rerun after a crash, timeout, or kernel
    restart skip everything that's already on disk instead of starting the whole RQ over."""
    csv_path = Path(csv_path)
    if not csv_path.exists():
        return set()
    with open(csv_path, newline='') as f:
        return {row[id_field] for row in csv.DictReader(f)}


def append_rows(csv_path, fieldnames, rows):
    """Appends rows to csv_path immediately and fsyncs to disk, writing the header only the first
    time. Called once per subject (not once per whole RQ), so a crash mid-run loses at most the
    subject(s) currently in flight, never the entire run's results."""
    if not rows:
        return
    csv_path = Path(csv_path)
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    with _csv_lock:
        write_header = not csv_path.exists()
        with open(csv_path, 'a', newline='') as f:
            w = csv.DictWriter(f, fieldnames=fieldnames)
            if write_header:
                w.writeheader()
            w.writerows(rows)
            f.flush()
            os.fsync(f.fileno())


def run_over_subjects(sids, per_subject_fn, desc=''):
    """Runs per_subject_fn(predictor_d, model_d, device, sid) for every sid in sids, with one
    worker thread per entry in WORKER_DEVICES pulling from a shared queue -- different subjects
    run on different GPUs concurrently. per_subject_fn must call append_rows() itself for every
    CSV it produces, as soon as that subject's rows are ready. Always prints this RQ's total wall
    time (and, for an actual run, the time/subject) so you can see where the budget is going."""
    sids = list(sids)
    t_start = time.time()
    if not sids:
        print(f'[{desc}] nothing to do -- every requested subject is already in the CSV(s) (0s)')
        return
    q = queue.Queue()
    for sid in sids:
        q.put(sid)
    remaining = len(sids)
    progress_lock = threading.Lock()

    def worker(device):
        nonlocal remaining
        predictor_d, model_d = PREDICTORS_BY_DEVICE[device], MODELS_BY_DEVICE[device]
        while True:
            try:
                sid = q.get_nowait()
            except queue.Empty:
                return
            t0 = time.time()
            try:
                per_subject_fn(predictor_d, model_d, device, sid)
                with progress_lock:
                    remaining -= 1
                    left = remaining
                print(f'[{desc}][{device}] {sid}: done in {time.time() - t0:.0f}s ({left} left)')
            except Exception:
                print(f'[{desc}][{device}] {sid}: FAILED')
                traceback.print_exc()
            finally:
                q.task_done()

    threads = [threading.Thread(target=worker, args=(d,), daemon=True) for d in WORKER_DEVICES]
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    elapsed = time.time() - t_start
    print(f'[{desc}] all {len(sids)} requested subjects processed in {elapsed:.0f}s '
          f'({elapsed / len(sids):.1f}s/subject, {len(WORKER_DEVICES)} GPU(s))')


## 9. Subjects for this run -- the first 50

`SUBJECTS_SUBSET` is the first 50 subjects out of every subject staged in the
manifest (loaded above as `subject_organs`).


In [ ]:
N_SUBJECTS = 50  # <-- change this one number to use more/fewer subjects

_all_loaded_sids = list(subject_organs)
assert N_SUBJECTS <= len(_all_loaded_sids), (
    f'requested {N_SUBJECTS} subjects, but only {len(_all_loaded_sids)} are loaded')

SUBJECTS_SUBSET = _all_loaded_sids[:N_SUBJECTS]
print(f'running the first {len(SUBJECTS_SUBSET)} of {len(_all_loaded_sids)} loaded subjects '
      f'-> {SUBJECTS_SUBSET}')


---
# RQ1 -- Do the 32 text-image fusion channels distinguish left and right anatomical structures?

**What I'm doing:** for every bilateral organ pair, crop one `predictor.patch_size` window
centred on the midpoint between the left and right organ centroids, run one forward pass per
prompt ("left X" vs. "right X"), and capture the literal **32-channel fusion tensor**
(`fusion_features` in `VoxTellDecoder.forward`) at each of the 4 intermediate U-Net decoder
stages, plus the finest (5th) stage's direct per-prompt logit (which has no 32-channel form --
see the note below). Then compare the left-prompt and right-prompt versions of these maps.

**Why:** the checkpoint config sets `num_heads=32` -- the transformer's fused text+image output
is literally projected into 32 channels and injected into the U-Net decoder at 5 stages. RQ1
asks whether *this exact quantity* differs for "left kidney" vs. "right kidney" -- not a proxy
for it, the actual tensor the model computes. If these 32 channels are nearly identical for
left vs. right, the U-Net decoder has nothing to work with to tell the two sides apart, no
matter what happens earlier in the pipeline.

**What I'll get (CSV columns):**
- `rq1_fusion_channel_similarity.csv`: one row per subject x organ pair x stage --
  `cosine_mean`/`cosine_min` (across the 32 channels), `corr_mean` (Pearson), `norm_l2_mean`
  (L2 difference normalized by the left map's own norm).
- `rq1_masks.csv`: GT and predicted mask (RLE) per organ pair x side, plus
  `dice_named_organ`/`dice_other_side_organ` (and IoU) -- the "other side" score is how you spot
  a laterality error directly (prediction overlapping the *wrong* organ's GT more than its own).
- `rq1_image_patches.csv`: the exact image patch each prediction ran on (base64), for offline
  overlay reconstruction.
- `rq1_organ_x_stage_mean_cosine.csv`: the aggregated organ x stage pivot table.

**What to plot:**
1. **Organ x stage heatmap** of `cosine_mean` (already built in section 10 below) -- low values
   (green in the heatmap) mark where/which organ's fusion channels discriminate laterality;
   near-1.0 (red) marks weak discrimination.
2. **Per-organ bar chart** ranking the 4 intermediate stages by `1 - cosine_mean` (discriminative
   power), to see if discrimination is concentrated at coarse or fine stages.
3. **Image + GT + prediction overlay**, reconstructed offline from `rq1_masks.csv` +
   `rq1_image_patches.csv`, for the organ with the *weakest* discrimination (highest
   `cosine_mean`) and the *strongest* (lowest), side by side.
4. **Scatter**: `cosine_mean` (x) vs. `dice_other_side_organ` (y) -- tests directly whether weak
   fusion-channel discrimination for an organ *predicts* an actual laterality error in the
   output mask, which is the real link RQ1's interpretation note is careful not to assume for free.

**Metrics:** cosine similarity (mean + min over 32 channels), Pearson correlation, normalized L2
difference (all per stage, per organ, per subject); Dice and IoU against both the named organ
and the mirrored organ's ground truth (to quantify laterality errors downstream of RQ1's own
finding).

**Only 4 of the 5 maskformer stages produce the literal 32-channel tensor** -- confirmed by
reading `voxtell/model/voxtell_model.py::VoxTellDecoder.forward`: the 5th (finest, full-res)
stage contracts the *unreshaped* mask embedding directly into the final per-prompt logit, with
no `num_heads` dimension. `capture_fusion_channels` below re-implements that forward loop
(copied, not imported -- there's no module boundary to hook the `fusion_features` local
variable) so it can read out the pure 32-channel tensor at the 4 stages where it exists, and
treats the 5th stage's logit as its own single-channel comparison object.

**If `voxtell` is ever upgraded, re-diff `capture_fusion_channels` against the installed
`voxtell/model/voxtell_model.py` before trusting its output.**

In [ ]:
import base64
import zlib

# These now also live in scripts/voxtell_mechanistic.py as vm.encode_mask_rle etc. -- imported
# directly here too so this section reads standalone.
encode_mask_rle = vm.encode_mask_rle
decode_mask_rle = vm.decode_mask_rle
encode_array_b64 = vm.encode_array_b64
decode_array_b64 = vm.decode_array_b64
print('RQ1 persistence helpers ready (encode_mask_rle, encode_array_b64, + decode_* counterparts)')


In [ ]:
import torch

@torch.inference_mode()
def capture_fusion_channels(decoder, skips, mask_embeddings):
    """Re-implements VoxTellDecoder.forward (voxtell v0.1.0) to read out `fusion_features` --
    the (B, 32, H, W, D) text-image fusion tensor RQ1 is literally about -- which is a local
    variable with no module boundary to hook.

    skips: from vm.encode(model, patch). mask_embeddings: the 5 tensors from
    model.project_to_decoder_channels, FINEST-to-COARSEST order (as that projection itself
    produces them -- same order vm.decode passes in).

    Returns:
      fusion_maps  : {stage_idx: (B, 32, H, W, D) float32 cpu} for the 4 INTERMEDIATE stages --
                     this IS the quantity RQ1's question is about.
      final_logits : (B, N, H, W, D) float32 cpu -- the 5th (finest) stage's direct per-prompt
                     logit. No 32-channel form exists at this stage (see markdown above), so
                     it's compared as its own single-channel map instead.
    """
    lres_input = skips[-1]
    mask_embeddings = list(mask_embeddings)[::-1]
    fusion_maps = {}
    final_logits = None
    n_stages = len(decoder.stages)
    for stage_idx in range(n_stages):
        x = decoder.transpconvs[stage_idx](lres_input)
        x = torch.cat((x, skips[-(stage_idx + 2)]), dim=1)
        x = decoder.stages[stage_idx](x)
        if stage_idx == n_stages - 1:
            # finest stage: direct per-prompt logit, NOT a 32-channel fusion map
            final_logits = torch.einsum('b c h w d, b n c -> b n h w d', x, mask_embeddings[-1]).float().cpu()
        elif stage_idx >= n_stages - len(mask_embeddings):
            # intermediate stage: this IS the literal 32-channel RQ1 fusion tensor
            mask_embedding = mask_embeddings.pop(0)
            b, _, c = mask_embedding.shape
            me_r = mask_embedding.view(b, decoder.num_heads, -1)
            fusion_features = torch.einsum('b c h w d, b n c -> b n h w d', x, me_r)
            fusion_maps[stage_idx] = fusion_features.float().cpu()
            x = torch.cat((x, fusion_features), dim=1)
        lres_input = x
    return fusion_maps, final_logits


@torch.inference_mode()
def fusion_channel_maps_for_prompt(model, patch_t, emb_row):
    """emb_row: (1, D) Qwen3 text embedding for ONE prompt. Runs encoder + 6-layer transformer
    decoder + final norm (same path VoxTellModel.forward itself takes) and returns the
    (fusion_maps, final_logits) pair documented in capture_fusion_channels above."""
    skips, memory = vm.encode(model, patch_t)
    tgt0 = model.project_text_embed(emb_row).unsqueeze(1)          # (1, 1, Q) -- the "layer 0" query (RQ3 reuses this)
    states, _ = vm.base_trajectory(model, tgt0, memory, model.pos_embed)
    mask_emb = model.transformer_decoder.norm(states[-1])          # (1, 1, Q) -- the "layer 6" / final query
    mes = [proj(mask_emb) for proj in model.project_to_decoder_channels]
    return capture_fusion_channels(model.decoder, skips, mes)


### RQ1 sanity check: print real shapes for one organ pair before trusting the full loop

In [ ]:
_check_sid = next(iter(subject_organs))
_person, _img_arr, _label_vol, _organs, _ids, _prompts = load_subject(_check_sid)
_organ = next(o for o in _organs if o.startswith('left'))
_right_organ = ve.mirror(_organ)
assert _right_organ in _organs, f'{_right_organ} missing for {_check_sid}'

_data, _bbox, _orig_shape = predictor.preprocess(_img_arr)
_crop_sl = tuple(slice(int(b[0]), int(b[1])) for b in _bbox)
_label_crop = _label_vol[_crop_sl]
_c_left = ve.centroid(_label_crop == _ids[_organ])
_c_right = ve.centroid(_label_crop == _ids[_right_organ])
assert _c_left is not None and _c_right is not None, 'organ pair not present in this subject'
_centre = tuple((a + b) / 2 for a, b in zip(_c_left, _c_right))
_patch, _offset = ve._crop_patch_around(_data, _centre, predictor.patch_size)
_patch_t = (_patch if torch.is_tensor(_patch) else torch.from_numpy(_patch)).to(predictor.device)

_emb_left = predictor.embed_text_prompts([_prompts[_organ]])[0].to(predictor.device)
_fusion_maps, _final_logits = fusion_channel_maps_for_prompt(model, _patch_t, _emb_left)

print('organ pair checked:', _organ, '/', _right_organ, 'in', _check_sid)
print('intermediate fusion-stage shapes (expect 4 entries, channel dim = 32):')
for k, v in sorted(_fusion_maps.items()):
    print(' stage', k, '->', tuple(v.shape))
print('finest-stage final_logits shape (expect channel dim = 1, N=1 prompt):', tuple(_final_logits.shape))
assert len(_fusion_maps) == 4, f'expected 4 intermediate fusion stages, got {len(_fusion_maps)}'
assert all(v.shape[1] == 32 for v in _fusion_maps.values()), 'expected 32 fusion channels per stage'
print('OK -- shapes match expectations. Proceed to the full loop below.')


### RQ1 full loop: every subject x every bilateral organ pair -> the 3 CSVs described above

In [ ]:
import numpy as np
import torch.nn.functional as F


def _flat(t):
    return t.reshape(t.shape[0], t.shape[1], -1)  # (B, C, N)


def channelwise_stats(left, right):
    """left, right: (1, C, H, W, D) float32 cpu -- one prompt's fusion map at one stage.
    Returns the per-channel-averaged cosine similarity, Pearson correlation, and normalized L2
    difference described in the RQ1 markdown above (low cosine / high L2 = strong laterality
    discrimination at this stage, for this organ)."""
    l, r = _flat(left)[0], _flat(right)[0]  # (C, N)
    cos = F.cosine_similarity(l, r, dim=1).numpy()
    l_c = l - l.mean(dim=1, keepdim=True)
    r_c = r - r.mean(dim=1, keepdim=True)
    denom = (l_c.norm(dim=1) * r_c.norm(dim=1)).clamp_min(1e-8)
    corr = ((l_c * r_c).sum(dim=1) / denom).numpy()
    l2 = (l - r).norm(dim=1)
    norm_l2 = (l2 / l.norm(dim=1).clamp_min(1e-8)).numpy()
    return {'cosine_mean': float(cos.mean()), 'cosine_min': float(cos.min()),
            'corr_mean': float(corr.mean()), 'norm_l2_mean': float(norm_l2.mean())}


def mask_from_logit(logit):
    """logit: (1, 1, H, W, D) float32 cpu -> (H, W, D) bool. Same threshold convention as
    predict_masks in scripts/voxtell_experiments.py (torch.sigmoid(logit) > 0.5)."""
    return (torch.sigmoid(logit[0, 0]) > 0.5).numpy()


rq1_sim_csv = RESULTS_ROOT / 'rq1' / 'rq1_fusion_channel_similarity.csv'
rq1_masks_csv = RESULTS_ROOT / 'rq1' / 'rq1_masks.csv'
rq1_patches_csv = RESULTS_ROOT / 'rq1' / 'rq1_image_patches.csv'

rq1_sim_fields = ['dataset', 'person', 'subject_id', 'organ_pair', 'stage', 'n_channels',
                   'cosine_mean', 'cosine_min', 'corr_mean', 'norm_l2_mean']
rq1_mask_fields = ['dataset', 'person', 'subject_id', 'organ_pair', 'side', 'label_id', 'prompt_text',
                    'patch_shape', 'n_gt_voxels', 'n_pred_voxels', 'dice_named_organ', 'iou_named_organ',
                    'dice_other_side_organ', 'iou_other_side_organ', 'gt_mask_rle', 'pred_mask_rle']
rq1_patch_fields = ['dataset', 'person', 'subject_id', 'organ_pair', 'patch_shape', 'patch_dtype',
                     'patch_offset', 'image_patch_b64']


def _rq1_one_subject(predictor_d, model_d, device, sid):
    sim_rows, mask_rows, patch_rows = [], [], []
    person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
    data, bbox, orig_shape = predictor_d.preprocess(img_arr)
    crop_sl = tuple(slice(int(b[0]), int(b[1])) for b in bbox)
    label_crop = label_vol[crop_sl]
    left_organs = sorted(o for o in organs if o.startswith('left') and ve.mirror(o) in organs)

    for organ in left_organs:
        # one organ pair = one laterality test case: same image, only the prompt's side word changes
        right_organ = ve.mirror(organ)
        organ_core = organ.split(None, 1)[1]
        c_left = ve.centroid(label_crop == ids[organ])
        c_right = ve.centroid(label_crop == ids[right_organ])
        if c_left is None or c_right is None:
            continue  # organ pair not present in this subject's scan
        centre = tuple((a + b) / 2 for a, b in zip(c_left, c_right))
        patch, offset = ve._crop_patch_around(data, centre, predictor_d.patch_size)
        patch_t = (patch if torch.is_tensor(patch) else torch.from_numpy(patch)).to(device)
        patch_shape = tuple(int(s) for s in predictor_d.patch_size)

        # ground truth for both sides, cropped into the SAME patch window as the prediction
        end = offset + np.array(patch_shape)
        crop_in_label = tuple(slice(max(int(o), 0), min(int(e), int(l)))
                               for o, e, l in zip(offset, end, label_crop.shape))
        paste_in_patch = tuple(slice(0, s.stop - s.start) for s in crop_in_label)

        def gt_in_patch(organ_name):
            gt = np.zeros(patch_shape, dtype=bool)
            gt[paste_in_patch] = (label_crop[crop_in_label] == ids[organ_name])
            return gt

        gt_left, gt_right = gt_in_patch(organ), gt_in_patch(right_organ)

        # the ONLY thing that differs between these two forward passes is the prompt's side word
        _emb_both = predictor_d.embed_text_prompts([prompts[organ], prompts[right_organ]])[0]  # (2, D)
        emb_left, emb_right = _emb_both[0:1].to(device), _emb_both[1:2].to(device)
        fusion_left, final_left = fusion_channel_maps_for_prompt(model_d, patch_t, emb_left)
        fusion_right, final_right = fusion_channel_maps_for_prompt(model_d, patch_t, emb_right)

        # RQ1's core comparison: left-prompt vs right-prompt 32-channel fusion map, per stage
        for stage_idx in sorted(fusion_left):
            stats = channelwise_stats(fusion_left[stage_idx], fusion_right[stage_idx])
            sim_rows.append({'dataset': DATASET_NAME, 'person': person, 'subject_id': sid,
                              'organ_pair': organ_core, 'stage': f'intermediate_{stage_idx}',
                              'n_channels': fusion_left[stage_idx].shape[1], **stats})

        # the finest stage's direct logit, as its own (1-channel) comparison -- see markdown note
        final_stats = channelwise_stats(final_left, final_right)
        sim_rows.append({'dataset': DATASET_NAME, 'person': person, 'subject_id': sid,
                          'organ_pair': organ_core, 'stage': 'finest_logit',
                          'n_channels': final_left.shape[1], **final_stats})

        # threshold each prompt's own prediction and score it against BOTH organs' GT --
        # high overlap with the WRONG organ's GT is a laterality error, visible directly below
        pred_left, pred_right = mask_from_logit(final_left), mask_from_logit(final_right)
        for side, pred, gt_named, gt_other, label_id, prompt_text in (
            ('left', pred_left, gt_left, gt_right, ids[organ], prompts[organ]),
            ('right', pred_right, gt_right, gt_left, ids[right_organ], prompts[right_organ]),
        ):
            mask_rows.append({
                'dataset': DATASET_NAME, 'person': person, 'subject_id': sid,
                'organ_pair': organ_core, 'side': side, 'label_id': label_id,
                'prompt_text': prompt_text, 'patch_shape': 'x'.join(map(str, patch_shape)),
                'n_gt_voxels': int(gt_named.sum()), 'n_pred_voxels': int(pred.sum()),
                'dice_named_organ': round(dice_score(pred, gt_named), 6),
                'iou_named_organ': round(iou_score(pred, gt_named), 6),
                'dice_other_side_organ': round(dice_score(pred, gt_other), 6),
                'iou_other_side_organ': round(iou_score(pred, gt_other), 6),
                'gt_mask_rle': encode_mask_rle(gt_named), 'pred_mask_rle': encode_mask_rle(pred),
            })

        # the exact image patch both forward passes ran on, for offline overlay reconstruction
        patch_np = patch_t[0, 0].detach().cpu().numpy()  # (H, W, D), drop batch+channel dims
        patch_rows.append({
            'dataset': DATASET_NAME, 'person': person, 'subject_id': sid, 'organ_pair': organ_core,
            'patch_shape': 'x'.join(map(str, patch_shape)), 'patch_dtype': 'float16',
            'patch_offset': 'x'.join(map(str, offset.astype(int).tolist())),
            'image_patch_b64': encode_array_b64(patch_np, dtype=np.float16),
        })

    append_rows(rq1_sim_csv, rq1_sim_fields, sim_rows)
    append_rows(rq1_masks_csv, rq1_mask_fields, mask_rows)
    append_rows(rq1_patches_csv, rq1_patch_fields, patch_rows)


_rq1_done = completed_ids(rq1_sim_csv)
_rq1_todo = [sid for sid in SUBJECTS_SUBSET if sid not in _rq1_done]
print(f'RQ1: {len(_rq1_done)} subjects already in {rq1_sim_csv.name}, {len(_rq1_todo)} left to run')
run_over_subjects(_rq1_todo, _rq1_one_subject, desc='rq1')


### Compress rq1_image_patches.csv (it's the only large RQ1 output -- base64 image patches,
only needed for offline overlay figures, not for the bias numbers above). Gzip it in place and
drop the raw CSV so Kaggle's Output only has to package/download the much smaller `.gz`.

In [ ]:
import gzip
import shutil

if rq1_patches_csv.exists():
    _gz_path = rq1_patches_csv.with_suffix(rq1_patches_csv.suffix + '.gz')
    _raw_size = rq1_patches_csv.stat().st_size
    with open(rq1_patches_csv, 'rb') as _f_in, gzip.open(_gz_path, 'wb', compresslevel=6) as _f_out:
        shutil.copyfileobj(_f_in, _f_out, length=64 * 1024 * 1024)  # 64MB chunks, avoid loading 4GB into RAM
    _gz_size = _gz_path.stat().st_size
    rq1_patches_csv.unlink()  # drop the raw CSV -- only the .gz needs to be downloaded
    print(f'{rq1_patches_csv.name}: {_raw_size / 1e9:.2f} GB -> {_gz_path.name}: {_gz_size / 1e9:.2f} GB '
          f'({_gz_size / _raw_size:.0%} of original)')
else:
    print(f'{rq1_patches_csv.name} does not exist yet -- run the RQ1 loop above first.')

### RQ1 aggregation: organ x stage similarity table (plot #1 above), CSV only

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rq1_df = pd.read_csv(rq1_sim_csv)
rq1_pivot = rq1_df.pivot_table(index='organ_pair', columns='stage', values='cosine_mean', aggfunc='mean')
rq1_pivot = rq1_pivot.reindex(columns=[c for c in
    ['intermediate_0', 'intermediate_1', 'intermediate_2', 'intermediate_3', 'finest_logit']
    if c in rq1_pivot.columns])
rq1_pivot.to_csv(RESULTS_ROOT / 'rq1' / 'rq1_organ_x_stage_mean_cosine.csv')
print('wrote', RESULTS_ROOT / 'rq1' / 'rq1_organ_x_stage_mean_cosine.csv')

fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(rq1_pivot))))
im = ax.imshow(rq1_pivot.values, aspect='auto', cmap='RdYlGn_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(rq1_pivot.columns)), rq1_pivot.columns, rotation=45, ha='right')
ax.set_yticks(range(len(rq1_pivot.index)), rq1_pivot.index)
ax.set_xlabel('decoder stage')
ax.set_title('RQ1: mean left/right fusion-channel cosine similarity\n(low = strong laterality discrimination at that stage)')
fig.colorbar(im, ax=ax, label='mean cosine similarity')
fig.tight_layout()
plt.show()  # inline only -- nothing saved to disk as an image, see the notebook-wide CSV policy


---
## Outputs -- RQ1 CSVs for 50 subjects

Masks are in the **patch** coordinate frame (`patch_shape` column) -- decode with
`vm.decode_mask_rle(cell, patch_shape_tuple)`.

| RQ | File | Columns worth knowing | Decode masks with |
|---|---|---|---|
| RQ1 | `rq1_fusion_channel_similarity.csv` | `cosine_mean`/`cosine_min`/`corr_mean`/`norm_l2_mean` per stage | -- |
| RQ1 | `rq1_masks.csv` | `dice_named_organ`/`dice_other_side_organ` (laterality error signal) | `vm.decode_mask_rle` |
| RQ1 | `rq1_image_patches.csv.gz` | base64 image patches, for offline overlay figures | `vm.decode_array_b64` |
| RQ1 | `rq1_organ_x_stage_mean_cosine.csv` | aggregated organ x stage pivot table | -- |

Rows are keyed by `subject_id`, so results from separate runs (e.g. different
`MAX_SUBJECTS` caps, or resumed after a Kaggle session timeout) can be combined with a plain
`pandas.concat` + `drop_duplicates(subset=['subject_id', ...])`.
